# A Python chatbot with a bounded agent loop
Core demo 50–80 minutes. Offline by default.
No SDK import or API call occurs in offline mode.

**Bridge from regression:** the first notebook predicted a number. An LLM generates tokens; a tool lets the application supply approved facts rather than rely on generated guesses. These are different models and tasks.

**Goal:** trace one tool call, then add and test your own FAQ topic in A4b.

## A1 • 50–52 min • Start with ordinary Python

Read FAQ['schedule'] first: it is just a dictionary lookup, with no model or agent loop. Expected: Mode: offline, Allowed tool: lookup_faq, and the schedule fact. Offline mode is scripted keyword matching, not AI inference. Open ../src/agent.py beside this notebook; inspect FAQ first, then dispatch in A2, and only then Chatbot.ask in A3.

In [1]:
import sys, json, os
from pathlib import Path
sys.path.insert(0, str(Path('../src').resolve()))
from agent import Chatbot, dispatch, TOOLS, FAQ
MODE = 'offline'  # Change to live only after configuring environment variables.
print('Mode:', MODE)
print('Allowed tool:', TOOLS[0]['function']['name'])
print('Plain Python lookup:', FAQ['schedule'])

Mode: offline
Allowed tool: lookup_faq
Plain Python lookup: Workshop: Monday October 26, 2026, 90 minutes. Start time is not specified.


## A2 • 52–55 min • Wrap the lookup in a validated JSON call

Expected: schedule answer, Unsupported topic error, Unknown tool error. The model proposes a function name and JSON arguments. Your application decides whether the call is valid. A schema describes the contract; validation also happens in Python.

In [2]:
print(json.loads(dispatch('lookup_faq', '{"topic":"schedule"}')))
print(json.loads(dispatch('lookup_faq', '{"topic":"unknown"}')))
print(json.loads(dispatch('delete_files', '{}')))

{'answer': 'Workshop: Monday October 26, 2026, 90 minutes. Start time is not specified.'}
{'error': 'Unsupported topic'}
{'error': 'Unknown tool'}


## A3 • 55–62 min • Complete a tool cycle

Expected trace: lookup_faq with topic schedule, followed by the schedule answer. Roles: system, user, assistant, tool, assistant. First assistant requests a tool, Python executes it, second assistant uses the result. Live wording can vary.

In [3]:
bot = Chatbot(mode=MODE)
print(bot.ask('When is the workshop?', trace=True))
print('Message roles:', [message['role'] for message in bot.messages])

TOOL lookup_faq {"topic": "schedule"} => {"answer": "Workshop: Monday October 26, 2026, 90 minutes. Start time is not specified."}
Workshop: Monday October 26, 2026, 90 minutes. Start time is not specified.
Message roles: ['system', 'user', 'assistant', 'tool', 'assistant']


## A4 • 62–67 min • Conversation and multiple tools

Expected two tool traces and the setup plus homework facts. Weather returns Ask about schedule, setup, or homework. History stays in this Python object, not a permanent database. Offline matching cannot resolve pronouns or paraphrases reliably; live inference may do so.

In [4]:
print(bot.ask('Tell me about setup and homework', trace=True))
print(bot.ask('What is the weather?'))
print('Conversation messages:', len(bot.messages))

TOOL lookup_faq {"topic": "setup"} => {"answer": "Use Python 3.11 or 3.12 and install requirements.txt in a virtual environment."}
TOOL lookup_faq {"topic": "homework"} => {"answer": "Add a new FAQ topic, test it, and explain one deployment choice."}
Use Python 3.11 or 3.12 and install requirements.txt in a virtual environment. Add a new FAQ topic, test it, and explain one deployment choice.
Ask about schedule, setup, or homework.
Conversation messages: 12


## A4b • 67–75 min • Your turn: add a contact topic

1. Open `../src/agent.py`. Inside `FAQ`, add this entry before `"schedule"`: `"contact": "Contact training@example.com for this fictional workshop.",`. Save the file.
2. Restart the kernel and rerun A1–A4 with MODE='offline'. Python caches imports; saving the module alone does not reload it.
3. Set RUN_EXTENSION=True below and run this cell. Expected: a contact tool trace and Contact extension passed.
4. Explain to a partner: you changed tool data and the derived schema, not model weights. No retraining was needed.

This exercise is skipped by default so Run All works before you edit the source. The address is fictional. Homework: add malformed-JSON and unsupported-topic checks.

In [5]:
RUN_EXTENSION = False  # Enable after editing the FAQ and restarting the kernel.
if RUN_EXTENSION:
    assert 'contact' in FAQ, 'Add contact in src/agent.py, save, and restart the kernel.'
    assert 'contact' in TOOLS[0]['function']['parameters']['properties']['topic']['enum']
    answer = Chatbot(mode='offline').ask('contact', trace=True)
    assert 'training@example.com' in answer
    print('Contact extension passed')
else:
    print('Contact exercise ready: follow A4b, then enable RUN_EXTENSION.')

Contact exercise ready: follow A4b, then enable RUN_EXTENSION.


## A5 • 75–80 min • Test boundaries

Expected: Empty input blocked; Agent stopped: tool-call round limit reached; Offline validation tests passed. The limit bounds model rounds, not wall-clock duration. Live calls also use a timeout and bounded retries.

In [6]:
assert 'error' in json.loads(dispatch('lookup_faq', 'not json'))
assert 'error' in json.loads(dispatch('lookup_faq', '{"topic":"schedule","extra":1}'))
assert 'error' in json.loads(dispatch('lookup_faq', '{"topic":[]}'))
try:
    bot.ask(' ')
except ValueError:
    print('Empty input blocked')
# Inject a fake model that never stops requesting tools.
def endless(messages):
    return {'role':'assistant','content':None,'tool_calls':[{'id':'loop','type':'function','function':{'name':'lookup_faq','arguments':'{"topic":"setup"}'}}]}
try:
    Chatbot(model=endless).ask('setup', max_rounds=2)
except RuntimeError as e:
    print(e)
print('Offline validation tests passed')

Empty input blocked
Agent stopped: tool-call round limit reached
Offline validation tests passed


## A6 • Optional live API demo • Outside the core timeline

Expected offline output: Live API skipped. No credentials or network required. Live output should contain the schedule fact, but wording and tool choices vary. Use a tool-capable model ID available to your account. No real credential is included or printed. See Setup.md.

In [7]:
# This cell stays offline unless MODE was deliberately changed above.
if MODE == 'live':
    try:
        print(Chatbot(mode='live').ask('What is the workshop schedule?', trace=True))
    except Exception as error:
        print('Live request failed:', type(error).__name__)
        print('Check key, model access, network, quota, and tool support. Use offline mode meanwhile.')
else:
    print('Live API skipped. No credentials or network required.')

Live API skipped. No credentials or network required.


## A7 • Optional interactive chatbot

Expected default: Interactive chat disabled; scripted examples above are complete. Interactive mode waits for typed questions until quit. Do not run this loop during automated testing.

In [8]:
# Set RUN_CHAT = True only for an interactive session. Run All never blocks.
RUN_CHAT = False
if RUN_CHAT:
    chat = Chatbot(mode=MODE)
    while True:
        question = input('You (quit to stop): ').strip()
        if question.lower() in {'quit','exit'}: break
        try:
            print('Agent:', chat.ask(question))
        except Exception as error:
            print('Request failed:', type(error).__name__, '- try offline mode or check setup.')
else:
    print('Interactive chat disabled; scripted examples above are complete.')

Interactive chat disabled; scripted examples above are complete.
